# Benchmark Gecode BAB + V1LD sur CCMKP

Ce notebook :
1. **Chronomètre une itération V1LD / V1CLD** sur l'instance choisie — sanity check avant de lancer le BAB
2. **Lance `bab_mkp_gecode`** (B&B Gecode avec borne V1LD à chaque nœud)
3. **Affiche les statistiques** : valeur, nœuds, temps, profondeur

Modifier la cellule **Paramètres** pour changer l'instance ou les options du solveur.

In [65]:
# ── Paramètres communs ────────────────────────────────────────────────────────
DATASET     = 'data/knapsack/datasets-song/ccmk-song-mknap1_4-S500.txt'
REPLICATION = 0
EPSILON     = 0.10

T_ROOT      = 200   # itérations V1LD/V1CLD au 1er appel (cold start)
T_NODE      = 5     # itérations aux nœuds suivants (warm-start, mode normal)
TIME_LIMIT  = -1    # limite de temps globale (secondes)

# Mode patience : si True, chaque nœud tourne jusqu'à convergence (tol=PATIENCE_TOL)
PATIENCE     = False
PATIENCE_TOL = 0.1   # tolérance de convergence (10^-1)

BAB_BIN  = 'solver_bab/build/bab_mkp_gecode'
T_BENCH  = 10

In [66]:
import os, sys, time, subprocess, re
import numpy as np

# VSCode Jupyter injecte __vsc_ipynb_file__ avec le chemin absolu du notebook.
# Fallback sur os.getcwd() si la variable n'existe pas (autre environnement).
try:
    ROOT = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    ROOT = os.getcwd()

# Les modules .so sont dans lib/ depuis le déplacement de la racine
LIB = os.path.join(ROOT, 'lib')
for p in [LIB, ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'ROOT : {ROOT}')
print(f'LIB  : {LIB}')
print(f'Python : {sys.version.split()[0]}')

import v1ld_generic_cpp as cpp
print(f'Module : {cpp.__file__}')

def load_song(filepath, replication=0):
    rows = []
    with open(filepath) as f:
        for line in f:
            v = list(map(float, line.strip().split(',')))
            if int(v[4]) != replication:
                continue
            m, n = int(v[2]), int(v[3])
            A = np.array(v[5 : 5+m*n]).reshape(m, n)
            b = np.array(v[5+m*n : 5+m*n+m])
            c = np.array(v[5+m*n+m : 5+m*n+m+n])
            rows.append((A, b, c))
    N = len(rows)
    assert N > 0, f'Aucun scénario pour replication={replication}'
    return (
        rows[0][2],
        np.ones(N) / N,
        np.stack([r[0] for r in rows]),
        np.stack([r[1] for r in rows]),
    )

c, p, A, b = load_song(os.path.join(ROOT, DATASET), REPLICATION)
N, m, n = A.shape
print(f'\nInstance : {os.path.basename(DATASET)}')
print(f'  n={n}, m={m}, N={N}, ε={EPSILON}, rep={REPLICATION}')

ROOT : /Users/roman.lendormy/Documents/Cours/X25 - 3A/Stage/Learning_bounds_CCCP
LIB  : /Users/roman.lendormy/Documents/Cours/X25 - 3A/Stage/Learning_bounds_CCCP/lib
Python : 3.11.9
Module : /Users/roman.lendormy/Documents/Cours/X25 - 3A/Stage/Learning_bounds_CCCP/lib/v1ld_generic_cpp.cpython-311-darwin.so

Instance : ccmk-song-mknap1_4-S500.txt
  n=20, m=10, N=500, ε=0.1, rep=0


## 1. Chrono : temps par itération V1LD / V1CLD

Mesure le coût d'**une itération de sous-gradient** avant de lancer le BAB.
Si `v1LD` est trop lent (>1 s/iter), le BAB avec ζ exact sera très coûteux.

In [67]:
prob_ld  = cpp.MKPProblemBnB(c, p, A, b, EPSILON)
prob_cld = cpp.MKPCLDProblem(c, p, A, b, EPSILON)

# Warmup (1 itération, résultats ignorés)
cpp.solve_v1ld( prob_ld,  T=1, log_interval=0)
cpp.solve_v1cld(prob_cld, T=1, log_interval=0)

# Chrono sur T_BENCH itérations
t0 = time.perf_counter()
r_ld = cpp.solve_v1ld(prob_ld, T=T_BENCH, log_interval=0)
ms_ld = (time.perf_counter() - t0) / T_BENCH * 1000

t0 = time.perf_counter()
r_cld = cpp.solve_v1cld(prob_cld, T=T_BENCH, log_interval=0)
ms_cld = (time.perf_counter() - t0) / T_BENCH * 1000

print(f'v1LD  (ζ exact B&B) : {ms_ld:8.1f} ms/iter')
print(f'v1CLD (ζ B&B 1D)    : {ms_cld:8.1f} ms/iter')
print(f'Ratio v1CLD/v1LD    : {ms_cld/ms_ld:.4f}x')
print()

# Estimation du coût du BAB
print(f'Estimation coût BAB (T_node={T_NODE} iters/nœud) :')
print(f'  v1LD  par nœud : {ms_ld * T_NODE:.0f} ms')
print(f'  v1CLD par nœud : {ms_cld * T_NODE:.0f} ms')

if ms_ld * T_NODE > 5000:
    print()
    print('  ⚠  v1LD >5 s/nœud — le BAB risque d\'être très lent.')
    print('     Envisager de réduire T_NODE ou d\'utiliser une instance plus petite.')

v1LD  (ζ exact B&B) :     75.4 ms/iter
v1CLD (ζ B&B 1D)    :      7.6 ms/iter
Ratio v1CLD/v1LD    : 0.1013x

Estimation coût BAB (T_node=5 iters/nœud) :
  v1LD  par nœud : 377 ms
  v1CLD par nœud : 38 ms


## 2. Lancement — V1CLD (ζ B&B 1D exact)

In [68]:
bin_path     = os.path.join(ROOT, BAB_BIN)
dataset_path = os.path.join(ROOT, DATASET)

if not os.path.isfile(bin_path):
    raise FileNotFoundError(f'Binaire introuvable : {bin_path}')

def run_bab(mode, patience=False, patience_tol=0.1):
    cmd = [bin_path, dataset_path, str(EPSILON), str(T_NODE),
           str(TIME_LIMIT), str(REPLICATION), str(T_ROOT), mode]
    if patience:
        cmd += ['patience', str(patience_tol)]
    t_wall = time.perf_counter()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    t_wall = time.perf_counter() - t_wall
    if proc.returncode != 0:
        raise RuntimeError(f'bab_mkp_gecode [{mode}] erreur {proc.returncode}\n{proc.stderr}')
    return proc.stdout, t_wall

def parse_result(out):
    def pf(pat, default=float('nan')):
        m = re.search(pat, out)
        return float(m.group(1)) if m else default
    def pi(pat, default=-1):
        m = re.search(pat, out)
        return int(m.group(1)) if m else default
    st = re.search(r'Statut\s*:\s*(\S+)', out)
    return dict(
        statut        = st.group(1) if st else '?',
        obj           = pf(r'Valeur optimale\s*:\s*([\d.]+)'),
        greedy        = pf(r'Incumbent initial.*?:\s*([\d.]+)'),
        best_ub       = pf(r'UB finale\s*:\s*([\d.]+)'),
        nodes         = pi(r'N.*uds Gecode\s*:\s*(\d+)'),
        naive_nodes   = pi(r'N.*uds Gecode\s*:\s*\d+ / (\d+)'),
        depth         = pi(r'Profondeur max\s*:\s*(\d+)'),
        depth_max     = pi(r'Profondeur max\s*:\s*\d+ / (\d+)'),
        fails         = pi(r'Échecs totaux\s*:\s*(\d+)'),
        fails_bound   = pi(r'dont borne.*?:\s*(\d+)'),
        fails_cp      = pi(r'dont propagation CP\s*:\s*(\d+)'),
        inc_imp       = pi(r'Améliorations incumbent\s*:\s*(\d+)'),
        more_calls    = pi(r'Appels more\(\).*?:\s*(\d+)'),
        ub_tight      = pi(r'UB V1LD utile.*?:\s*(\d+)'),
        t_solve       = pf(r'Temps total\s*:\s*([\d.]+)'),
        t_root_bound  = pf(r'Temps borne root\s*:\s*([\d.]+)'),
        t_bound_total = pf(r'Temps borne total\s*:\s*([\d.]+)'),
        t_bab_pure    = pf(r'Temps BAB pur\s*:\s*([\d.]+)'),
        ms_node       = pf(r'Temps moyen par nœud\s*:\s*([\d.]+)'),
    )

print('Lancement V1CLD...', flush=True)
out_v1cld, wall_v1cld = run_bab('v1cld')
r_v1cld = parse_result(out_v1cld)
print(f"  obj={r_v1cld['obj']:.1f}  nœuds={r_v1cld['nodes']}  "
      f"t={r_v1cld['t_solve']:.2f}s  t_borne={r_v1cld['t_bound_total']:.2f}s  "
      f"t_bab_pur={r_v1cld['t_bab_pure']:.2f}s  [{r_v1cld['statut']}]")

Lancement V1CLD...
  obj=5880.0  nœuds=14305  t=994.46s  t_borne=864.34s  t_bab_pur=130.10s  [OPTIMAL]


## 3. Résumé V1CLD

In [69]:
def display_result(r, mode_label):
    obj, greedy = r['obj'], r['greedy']
    nodes, naive = r['nodes'], r['naive_nodes']
    pct_naive   = 100.0 * nodes / naive if naive > 0 else float('nan')
    fails       = r['fails']
    pct_b  = 100.0 * r['fails_bound'] / fails if fails > 0 else float('nan')
    pct_cp = 100.0 * r['fails_cp']    / fails if fails > 0 else float('nan')
    pct_ub = 100.0 * r['ub_tight']    / r['more_calls'] if r['more_calls'] > 0 else float('nan')
    gap_greedy = 100.0 * (obj - greedy)       / obj if obj > 0 else float('nan')
    gap_final  = 100.0 * (r['best_ub'] - obj) / obj if obj > 0 else float('nan')
    pct_root   = 100.0 * r['t_root_bound']  / r['t_solve'] if r['t_solve'] > 0 else float('nan')
    pct_borne  = 100.0 * r['t_bound_total'] / r['t_solve'] if r['t_solve'] > 0 else float('nan')
    pct_bab    = 100.0 * r['t_bab_pure']    / r['t_solve'] if r['t_solve'] > 0 else float('nan')

    print(f'┌──────────────────────────────────────────────────┐')
    print(f'│  Mode : {mode_label:<43}│')
    print(f'│  Instance : {os.path.basename(DATASET):<37}│'[:52] + '│')
    print(f'│  n={n}, m={m}, N={N}, ε={EPSILON}, rep={REPLICATION}                    │'[:52] + '│')
    print(f'│  T_root={T_ROOT}, T_node={T_NODE}, time_limit={TIME_LIMIT:.0f}s          │'[:52] + '│')
    print(f'├──────────────────────────────────────────────────┤')
    print(f'│  Statut          : {r["statut"]:<31}│')
    print(f'│  Valeur optimale : {obj:<31.4f}│')
    print(f'│  Greedy (init)   : {greedy:<6.1f}  (gap : {gap_greedy:.1f}%)                │'[:52] + '│')
    print(f'│  UB finale       : {r["best_ub"]:<6.1f}  (gap dual : {gap_final:.1f}%)            │'[:52] + '│')
    print(f'├──────────────────────────────────────────────────┤')
    print(f'│  Nœuds           : {nodes:<10}  ({pct_naive:.3f}% du naïf 2^n)    │'[:52] + '│')
    print(f'│  Profondeur max  : {r["depth"]} / {r["depth_max"]} théorique                   │'[:52] + '│')
    print(f'├──────────────────────────────────────────────────┤')
    print(f'│  Élagages totaux : {fails:<31}│')
    print(f'│    borne         : {r["fails_bound"]:<5}  ({pct_b:.1f}%)                       │'[:52] + '│')
    print(f'│    propag. CP    : {r["fails_cp"]:<5}  ({pct_cp:.1f}%)                       │'[:52] + '│')
    print(f'├──────────────────────────────────────────────────┤')
    print(f'│  Amél. incumbent : {r["inc_imp"]:<31}│')
    print(f'│  Appels more()   : {r["more_calls"]:<31}│')
    print(f'│  UB utile        : {r["ub_tight"]:<5}  ({pct_ub:.1f}% des appels)           │'[:52] + '│')
    print(f'├──────────────────────────────────────────────────┤')
    print(f'│  Temps total     : {r["t_solve"]:<31.3f}│')
    print(f'│  Temps borne root: {r["t_root_bound"]:<6.3f}  ({pct_root:.1f}% du total)            │'[:52] + '│')
    print(f'│  Temps borne tot.: {r["t_bound_total"]:<6.3f}  ({pct_borne:.1f}% du total)            │'[:52] + '│')
    print(f'│  Temps BAB pur   : {r["t_bab_pure"]:<6.3f}  ({pct_bab:.1f}% du total)            │'[:52] + '│')
    print(f'│  ms / nœud       : {r["ms_node"]:<31.1f}│')
    print(f'└──────────────────────────────────────────────────┘')

display_result(r_v1cld, 'V1CLD (ζ B&B 1D exact)')

┌──────────────────────────────────────────────────┐
│  Mode : V1CLD (ζ B&B 1D exact)                     │
│  Instance : ccmk-song-mknap1_4-S500.txt          ││
│  n=20, m=10, N=500, ε=0.1, rep=0                  │
│  T_root=200, T_node=5, time_limit=-1s          ││
├──────────────────────────────────────────────────┤
│  Statut          : OPTIMAL                        │
│  Valeur optimale : 5880.0000                      │
│  Greedy (init)   : 5410.0  (gap : 8.0%)           │
│  UB finale       : 6960.6  (gap dual : 18.4%)     │
├──────────────────────────────────────────────────┤
│  Nœuds           : 14305       (1.364% du naïf 2^n│
│  Profondeur max  : 53 / 520 théorique             │
├──────────────────────────────────────────────────┤
│  Élagages totaux : 14285                          │
│    borne         : 877    (6.1%)                  │
│    propag. CP    : 13408  (93.9%)                 │
├──────────────────────────────────────────────────┤
│  Amél. incumbent : 21           

## 4. Lancement — V1LD (ζ exact B&B)

In [70]:
print('Lancement V1LD...', flush=True)
out_v1ld, wall_v1ld = run_bab('v1ld')
r_v1ld = parse_result(out_v1ld)
print(f"  obj={r_v1ld['obj']:.1f}  nœuds={r_v1ld['nodes']}  t={r_v1ld['t_solve']:.2f}s  [{r_v1ld['statut']}]")

Lancement V1LD...


  obj=5880.0  nœuds=4156  t=228.19s  [OPTIMAL]


## 5. Résumé V1LD

In [71]:
display_result(r_v1ld, 'V1LD (ζ exact B&B)')

┌──────────────────────────────────────────────────┐
│  Mode : V1LD (ζ exact B&B)                         │
│  Instance : ccmk-song-mknap1_4-S500.txt          ││
│  n=20, m=10, N=500, ε=0.1, rep=0                  │
│  T_root=200, T_node=5, time_limit=-1s          ││
├──────────────────────────────────────────────────┤
│  Statut          : OPTIMAL                        │
│  Valeur optimale : 5880.0000                      │
│  Greedy (init)   : 5410.0  (gap : 8.0%)           │
│  UB finale       : 6473.3  (gap dual : 10.1%)     │
├──────────────────────────────────────────────────┤
│  Nœuds           : 4156        (0.396% du naïf 2^n│
│  Profondeur max  : 50 / 520 théorique             │
├──────────────────────────────────────────────────┤
│  Élagages totaux : 4136                           │
│    borne         : 1006   (24.3%)                 │
│    propag. CP    : 3130   (75.7%)                 │
├──────────────────────────────────────────────────┤
│  Amél. incumbent : 21           

## 6. Comparaison V1LD vs V1CLD

In [72]:
import pandas as pd

def ratio(a, b):
    return f'{a/b:.1f}x' if b and b != 0 else 'N/A'

rows = []
for label, r in [('V1LD', r_v1ld), ('V1CLD', r_v1cld)]:
    rows.append({
        'Mode'           : label,
        'Statut'         : r['statut'],
        'Valeur opt.'    : f"{r['obj']:.1f}",
        'Nœuds'          : r['nodes'],
        '% naïf'         : f"{100*r['nodes']/r['naive_nodes']:.3f}%" if r['naive_nodes'] > 0 else 'N/A',
        'Élagages borne' : f"{r['fails_bound']} ({100*r['fails_bound']/r['fails']:.0f}%)" if r['fails'] else 'N/A',
        'Élagages CP'    : f"{r['fails_cp']} ({100*r['fails_cp']/r['fails']:.0f}%)" if r['fails'] else 'N/A',
        'Amél. incumb.'  : r['inc_imp'],
        'Temps (s)'      : f"{r['t_solve']:.2f}",
        't_root (s)'     : f"{r['t_root_bound']:.3f}",
        'ms / nœud'      : f"{r['ms_node']:.1f}",
    })

df = pd.DataFrame(rows).set_index('Mode')
print(df.to_string())

print()
ld, cld = r_v1ld, r_v1cld
print(f"Nœuds    V1LD / V1CLD : {ratio(cld['nodes'], ld['nodes'])} plus de nœuds pour V1CLD")
print(f"Temps    V1LD / V1CLD : {ratio(cld['t_solve'], ld['t_solve'])} pour V1CLD")
print(f"ms/nœud  V1LD / V1CLD : {ratio(ld['ms_node'], cld['ms_node'])} plus cher par nœud pour V1LD")
print(f"t_root   V1LD / V1CLD : {ratio(ld['t_root_bound'], cld['t_root_bound'])} pour V1LD  (T_root={T_ROOT} iters)")
if ld['statut'] == 'OPTIMAL' and cld['statut'] == 'OPTIMAL':
    print(f"Même valeur optimale  : {ld['obj'] == cld['obj']}")

        Statut Valeur opt.  Nœuds  % naïf Élagages borne  Élagages CP  Amél. incumb. Temps (s) t_root (s) ms / nœud
Mode                                                                                                               
V1LD   OPTIMAL      5880.0   4156  0.396%     1006 (24%)   3130 (76%)             21    228.19     25.512      54.9
V1CLD  OPTIMAL      5880.0  14305  1.364%       877 (6%)  13408 (94%)             21    994.46      3.727      69.5

Nœuds    V1LD / V1CLD : 3.4x plus de nœuds pour V1CLD
Temps    V1LD / V1CLD : 4.4x pour V1CLD
ms/nœud  V1LD / V1CLD : 0.8x plus cher par nœud pour V1LD
t_root   V1LD / V1CLD : 6.8x pour V1LD  (T_root=200 iters)
Même valeur optimale  : True


## 7. Lancement — V1CLD mode Patience (convergence à tol=PATIENCE_TOL)

In [73]:
print(f'Lancement V1CLD patience (tol={PATIENCE_TOL})...', flush=True)
out_pat, wall_pat = run_bab('v1cld', patience=True, patience_tol=PATIENCE_TOL)
r_pat = parse_result(out_pat)
print(f"  obj={r_pat['obj']:.1f}  nœuds={r_pat['nodes']}  "
      f"t={r_pat['t_solve']:.2f}s  t_borne={r_pat['t_bound_total']:.2f}s  "
      f"t_bab_pur={r_pat['t_bab_pure']:.2f}s  [{r_pat['statut']}]")

Lancement V1CLD patience (tol=0.1)...
  obj=5880.0  nœuds=4101  t=5049.04s  t_borne=5003.32s  t_bab_pur=45.70s  [OPTIMAL]


## 8. Comparaison Normal / Patience / V1LD

In [74]:
import pandas as pd

def ratio(a, b):
    return f'{a/b:.2f}x' if b and b != 0 else 'N/A'

configs = [
    ('V1CLD normal',   r_v1cld),
    ('V1CLD patience', r_pat),
    ('V1LD',           r_v1ld),
]

rows = []
for label, r in configs:
    rows.append({
        'Mode'            : label,
        'Statut'          : r['statut'],
        'Valeur opt.'     : f"{r['obj']:.1f}",
        'Nœuds'           : r['nodes'],
        'Élag. borne'     : f"{r['fails_bound']} ({100*r['fails_bound']/r['fails']:.0f}%)" if r['fails'] else 'N/A',
        'Temps total (s)' : f"{r['t_solve']:.2f}",
        'Temps borne (s)' : f"{r['t_bound_total']:.2f}  ({100*r['t_bound_total']/r['t_solve']:.0f}%)" if r['t_solve'] else 'N/A',
        'Temps BAB pur (s)': f"{r['t_bab_pure']:.2f}  ({100*r['t_bab_pure']/r['t_solve']:.0f}%)" if r['t_solve'] else 'N/A',
        'ms / nœud'       : f"{r['ms_node']:.1f}",
    })

df = pd.DataFrame(rows).set_index('Mode')
print(df.to_string())

print()
print('--- Ratios vs V1CLD normal ---')
ref = r_v1cld
for label, r in configs[1:]:
    print(f'{label:<20}  nœuds: {ratio(r["nodes"], ref["nodes"])}  '
          f'bab_pur: {ratio(r["t_bab_pure"], ref["t_bab_pure"])}  '
          f't_total: {ratio(r["t_solve"], ref["t_solve"])}')

                 Statut Valeur opt.  Nœuds Élag. borne Temps total (s) Temps borne (s) Temps BAB pur (s) ms / nœud
Mode                                                                                                              
V1CLD normal    OPTIMAL      5880.0  14305    877 (6%)          994.46   864.34  (87%)     130.10  (13%)      69.5
V1CLD patience  OPTIMAL      5880.0   4101  1192 (29%)         5049.04  5003.32  (99%)       45.70  (1%)    1231.2
V1LD            OPTIMAL      5880.0   4156  1006 (24%)          228.19   199.05  (87%)      29.10  (13%)      54.9

--- Ratios vs V1CLD normal ---
V1CLD patience        nœuds: 0.29x  bab_pur: 0.35x  t_total: 5.08x
V1LD                  nœuds: 0.29x  bab_pur: 0.22x  t_total: 0.23x
